# CDK20_HUMAN: 相同蛋白の探索と既知活性化合物の収集

CDK20_HUMANはまだX線結晶構造が解かれていない。そこで、

1. CDK20自体の既知の活性化合物をChEMBLから収集する
2. UniProt/BLASTでCDK20に配列が近い蛋白を探す
3. 見つかった蛋白ごとにPDBエントリ数・ChEMBL活性化合物数を集計し、この研究で参照蛋白として活用できそうなものをランキングする


In [1]:
from pathlib import Path

TARGET = "CDK20_HUMAN"
OUTDIR = Path("data/cdk20_investigation")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ChEMBL Web APIが障害等で使えない場合のフォールバック(ChEMBL公式配布のSQLite)
CHEMBL_DB = Path("data/chembl/chembl_36.db")

In [2]:
from idmap import resolve_uniprot_accession

accession = resolve_uniprot_accession(TARGET)
print(accession)

12:32:37 [idmap.identifiers] Resolving UniProt entry name CDK20_HUMAN -> accession ...
12:32:38 [idmap.identifiers]   -> Q8IZL9


Q8IZL9


## 1. CDK20自体の既知活性化合物を収集(ChEMBL)

構造とは独立に、CDK20_HUMANに対する既知の活性化合物(pChEMBL値あり)をChEMBLから取得する。


In [3]:
import pandas as pd
from chembl import local as chembl_local

chembl_target_id = chembl_local.resolve_target_chembl_id(accession, CHEMBL_DB)
activities = chembl_local.fetch_activities(chembl_target_id, CHEMBL_DB)

compounds_df = pd.DataFrame(activities)
if not compounds_df.empty:
    compounds_df = compounds_df[
        ["molecule_chembl_id", "molecule_pref_name", "canonical_smiles",
         "standard_type", "standard_value", "standard_units", "pchembl_value",
         "assay_chembl_id", "document_chembl_id"]
    ].sort_values("pchembl_value", ascending=False).reset_index(drop=True)
compounds_df.to_csv(OUTDIR / "known_compounds.csv", index=False)
compounds_df

12:32:46 [chembl.local] Resolving UniProt accession Q8IZL9 -> ChEMBL target id (local DB) ...
12:32:46 [chembl.local]   -> CHEMBL3559690
12:32:46 [chembl.local] Fetching activities for target CHEMBL3559690 (pChEMBL value required, local DB) ...
12:32:46 [chembl.local]   -> 1 activities fetched for CHEMBL3559690


,molecule_chembl_id,molecule_pref_name,canonical_smiles,standard_type,standard_value,standard_units,pchembl_value,assay_chembl_id,document_chembl_id
0,CHEMBL4445812,None,O=C(Nc1ccc2cc(C(=O)Nc3nccs3)ccc2c1)c1ccc(Cl)c(...,Kd,8020,nM,5.1,CHEMBL4375310,CHEMBL4373706


In [4]:
from uniprot.entry import fetch_fasta

fasta = fetch_fasta(accession).decode()
sequence = "".join(line for line in fasta.splitlines() if not line.startswith(">"))
print(fasta)
print(f"sequence length: {len(sequence)}")

12:32:48 [uniprot.entry] Fetching UniProt FASTA for Q8IZL9 ...


>sp|Q8IZL9|CDK20_HUMAN Cyclin-dependent kinase 20 OS=Homo sapiens OX=9606 GN=CDK20 PE=1 SV=1
MDQYCILGRIGEGAHGIVFKAKHVETGEIVALKKVALRRLEDGFPNQALREIKALQEMED
NQYVVQLKAVFPHGGGFVLAFEFMLSDLAEVVRHAQRPLAQAQVKSYLQMLLKGVAFCHA
NNIVHRDLKPANLLISASGQLKIADFGLARVFSPDGSRLYTHQVATRWYRAPELLYGARQ
YDQGVDLWSVGCIMGELLNGSPLFPGKNDIEQLCYVLRILGTPNPQVWPELTELPDYNKI
SFKEQVPMPLEEVLPDVSPQALDLLGQFLLYPPHQRIAASKALLHQYFFTAPLPAHPSEL
PIPQRLGGPAPKAHPGPPHIHDFHVDRPLEESLLNPELIRPFILEG

sequence length: 346


## 2. UniProt/BLASTでCDK20に近い蛋白を探す

swissprot(UniProtKB/Swiss-Prot、審査済みエントリ)データベースに対し、Homo sapiensに限定してBLASTサーチを行い、
CDK20_HUMANに配列が近い蛋白(UniProt accession)を集める。配列同一性(identity)が40%を割る蛋白は
遠縁とみなし対象から除外する。


In [5]:
import pickle
from blastsearch import fetch_hits, submit_blast, wait_for_blast

HITS_CACHE = OUTDIR / "blast_hits.pkl"
RID_CACHE = OUTDIR / "blast_rid.txt"

if HITS_CACHE.exists():
    print(f"Using cached BLAST hits: {HITS_CACHE}")
    with open(HITS_CACHE, "rb") as f:
        hits = pickle.load(f)
else:
    if RID_CACHE.exists():
        rid = RID_CACHE.read_text().strip()
        print(f"Resuming existing BLAST job {rid} (submitted previously) ...")
    else:
        rid = submit_blast(sequence, program="blastp", database="swissprot", entrez_query="Homo sapiens[Organism]")
        RID_CACHE.write_text(rid)

    try:
        wait_for_blast(rid, poll_interval=10.0, timeout=600.0)
    except RuntimeError:
        # ジョブ自体が失敗した場合は再投函が必要なのでRIDキャッシュを破棄する
        RID_CACHE.unlink(missing_ok=True)
        raise
    # TimeoutErrorはRID_CACHEを残したまま伝播させる(セルを再実行すれば同じRIDで待機を再開できる)

    hits = fetch_hits(rid)
    with open(HITS_CACHE, "wb") as f:
        pickle.dump(hits, f)
    RID_CACHE.unlink(missing_ok=True)
    print(f"Saved BLAST hits to {HITS_CACHE}")

Using cached BLAST hits: data/cdk20_investigation/blast_hits.pkl


In [6]:
import pandas as pd
from blastsearch import parse_uniprot_subject_id
from uniprot.entry import fetch_entry_names

# UniProt accessionごとに最良ヒット(evalue最小)だけを残す。CDK20_HUMAN自身は除く。
best_by_accession = {}
for h in hits:
    acc = parse_uniprot_subject_id(h["subject_id"])
    if acc == accession:
        continue
    if acc not in best_by_accession or h["evalue"] < best_by_accession[acc]["evalue"]:
        best_by_accession[acc] = {**h, "accession": acc}
unique_hits = sorted(best_by_accession.values(), key=lambda h: h["evalue"])

hits_df = pd.DataFrame(unique_hits)

IDENTITY_MIN = 40.0
hits_df = hits_df[hits_df["identity"] >= IDENTITY_MIN].reset_index(drop=True)

entry_names = fetch_entry_names(hits_df["accession"].tolist())
hits_df["entry_name"] = hits_df["accession"].map(entry_names)
hits_df["coverage"] = (hits_df["align_length"] / len(sequence) * 100).round(1)
hits_df = hits_df[["accession", "entry_name", "identity", "coverage", "align_length", "evalue", "bit_score"]]
hits_df.to_csv(OUTDIR / "blast_hits.csv", index=False)
print(f"unique candidate proteins (identity >= {IDENTITY_MIN}%): {len(hits_df)}")


def _format_evalue(e: float) -> str:
    """0はそのまま「0」、それ以外は有効数字2桁の指数表記(例: 9.47e-95 -> 9.5e-95)にする。"""
    return "0" if e == 0 else f"{e:.1e}"


display_df = hits_df.head(20).copy()
display_df["identity"] = display_df["identity"].round(2)
display_df["bit_score"] = display_df["bit_score"].round().astype(int)
display_df["evalue"] = display_df["evalue"].apply(_format_evalue)
display_df


12:32:55 [uniprot.entry] Fetching UniProt entry names for 9 accessions ...
12:32:56 [uniprot.entry]   -> 9 entry names resolved


unique candidate proteins (identity >= 40.0%): 9


,accession,entry_name,identity,coverage,align_length,evalue,bit_score
0,Q00526,CDK3_HUMAN,45.28,88.7,307,6.8e-86,261
1,P06493,CDK1_HUMAN,43.10,83.8,290,5.0e-80,246
2,P50613,CDK7_HUMAN,43.14,88.4,306,6.8e-80,247
3,P24941,CDK2_HUMAN,43.77,85.8,297,1.1e-79,245
4,Q00535,CDK5_HUMAN,46.02,83.5,289,5.6e-79,243
5,P21127,CD11B_HUMAN,42.16,88.4,306,3.2e-73,241
6,Q9UQ88,CD11A_HUMAN,41.83,88.4,306,7.3e-73,240
7,P50750,CDK9_HUMAN,40.00,89.6,310,1.2e-68,219
8,O76039,CDKL5_HUMAN,40.00,85.3,295,2.8e-60,207


## 3. 見つかった蛋白ごとにPDBエントリ数・ChEMBL化合物数を集計してランキングする

上位ヒットについて、UniProtのPDB相互参照からPDBエントリ数を、ChEMBLから既知活性化合物数(ユニークな化合物数)を
集計する。両方が揃っている蛋白ほど、この研究(ドッキングテンプレート+SAR参照)で活用しやすいと考えられる。


In [7]:
TOP_N = 40
candidates_df = hits_df.head(TOP_N).copy()

In [9]:
import pickle

import requests
from uniprot.entry import fetch_protein_info
from chembl import local as chembl_local

RANKING_CACHE = OUTDIR / "protein_ranking.csv"
PDB_STRUCTURES_CACHE = OUTDIR / "pdb_structures.pkl"

if RANKING_CACHE.exists() and PDB_STRUCTURES_CACHE.exists():
    print(f"Using cached ranking: {RANKING_CACHE}")
    ranking_df = pd.read_csv(RANKING_CACHE)
    with open(PDB_STRUCTURES_CACHE, "rb") as f:
        pdb_structures_by_accession = pickle.load(f)
else:
    extra_rows = []
    pdb_structures_by_accession = {}
    for _, row in candidates_df.iterrows():
        acc = row["accession"]
        try:
            info = fetch_protein_info(acc)
            pdb_count = len(info["pdb_structures"])
            pdb_structures_by_accession[acc] = info["pdb_structures"]
        except requests.exceptions.RequestException as e:
            print(f"  [skip] {acc}: failed to fetch UniProt info ({e})")
            continue

        chembl_target_id = None
        try:
            chembl_target_id = chembl_local.resolve_target_chembl_id(acc, CHEMBL_DB)
            compounds = chembl_local.fetch_activities(chembl_target_id, CHEMBL_DB)
            compound_count = len({a["molecule_chembl_id"] for a in compounds})
        except ValueError:
            compound_count = 0  # ChEMBL target(SINGLE PROTEIN)が見つからない

        extra_rows.append({
            "accession": acc,
            "pdb_count": pdb_count,
            "chembl_target_id": chembl_target_id,
            "compound_count": compound_count,
        })

    extra_df = pd.DataFrame(extra_rows)
    ranking_df = candidates_df.merge(extra_df, on="accession").sort_values(
        ["pdb_count", "compound_count"], ascending=False
    ).reset_index(drop=True)
    ranking_df.to_csv(RANKING_CACHE, index=False)
    with open(PDB_STRUCTURES_CACHE, "wb") as f:
        pickle.dump(pdb_structures_by_accession, f)
    print(f"Saved ranking to {RANKING_CACHE}")

Using cached ranking: data/cdk20_investigation/protein_ranking.csv


In [16]:
ranking_display_df = ranking_df[
    ["accession", "entry_name", "chembl_target_id", "identity", "coverage",
     "align_length", "evalue", "bit_score", "pdb_count", "compound_count"]
].copy()
ranking_display_df["identity"] = ranking_display_df["identity"].round(2)
ranking_display_df["bit_score"] = ranking_display_df["bit_score"].round().astype(int)
ranking_display_df["evalue"] = ranking_display_df["evalue"].apply(_format_evalue)
ranking_display_df

,accession,entry_name,chembl_target_id,identity,coverage,align_length,evalue,bit_score,pdb_count,compound_count
0,P24941,CDK2_HUMAN,CHEMBL301,43.77,85.8,297,1.1e-79,245,512,2618
1,P50613,CDK7_HUMAN,CHEMBL3055,43.14,88.4,306,6.8e-80,247,53,627
2,P50750,CDK9_HUMAN,CHEMBL3116,40.00,89.6,310,1.2e-68,219,28,1741
3,P06493,CDK1_HUMAN,CHEMBL308,43.10,83.8,290,5.0e-80,246,13,1475
4,Q00535,CDK5_HUMAN,CHEMBL4036,46.02,83.5,289,5.6e-79,243,10,717
5,O76039,CDKL5_HUMAN,CHEMBL1163112,40.00,85.3,295,2.8e-60,207,3,21
6,Q00526,CDK3_HUMAN,CHEMBL4442,45.28,88.7,307,6.8e-86,261,2,47
7,P21127,CD11B_HUMAN,CHEMBL5808,42.16,88.4,306,3.2e-73,241,2,20
8,Q9UQ88,CD11A_HUMAN,CHEMBL5416,41.83,88.4,306,7.3e-73,240,0,48


## 4. 解像度2Å未満のX線構造を一括ダウンロードする

セクション2で集計したPDBエントリのうち、実験手法がX線結晶構造解析(`method == "X-ray"`)かつ
解像度2Å未満のものをすべてダウンロードする。蛋白(entry_name)ごとにサブディレクトリへ保存し、
既にファイルが存在する場合はスキップするので、繰り返し実行しても差分だけ取得される。


In [17]:
from rcsb import fetch_structure
from uniprot.entry import parse_resolution

RESOLUTION_CUTOFF = 2.0
STRUCT_DIR = OUTDIR / "structures"

entry_name_by_accession = candidates_df.set_index("accession")["entry_name"].to_dict()

# (accession, entry_name, pdb_id, resolution) のリストに集約(解像度の良い順)
targets = []
for acc, structures in pdb_structures_by_accession.items():
    for s in structures:
        if s["method"] != "X-ray":
            continue
        res = parse_resolution(s["resolution"])
        if res is None or res >= RESOLUTION_CUTOFF:
            continue
        targets.append((acc, entry_name_by_accession.get(acc, acc), s["id"], res))
targets.sort(key=lambda t: t[3])

total = len(targets)
print(f"X-ray structures with resolution < {RESOLUTION_CUTOFF} A: {total}")

downloaded = 0
skipped = 0
for i, (acc, entry_name, pdb_id, res) in enumerate(targets, start=1):
    subdir = STRUCT_DIR / entry_name
    subdir.mkdir(parents=True, exist_ok=True)
    output = subdir / f"{pdb_id}.cif"
    if output.exists():
        print(f"[{i}/{total}] {entry_name}/{pdb_id}: already exists, skipping")
        skipped += 1
        continue
    print(f"[{i}/{total}] {entry_name}/{pdb_id}: downloading (resolution={res:.2f} A) ...")
    fetch_structure(pdb_id, output)
    downloaded += 1

print(f"Done: {downloaded} downloaded, {skipped} already present, {total} total, saved under {STRUCT_DIR}")

X-ray structures with resolution < 2.0 A: 245
[1/245] CDK2_HUMAN/6Q4G: already exists, skipping
[2/245] CDK2_HUMAN/6Q49: already exists, skipping
[3/245] CDK2_HUMAN/6Q4H: already exists, skipping
[4/245] CDK2_HUMAN/6Q48: already exists, skipping
[5/245] CDK2_HUMAN/6Q4J: already exists, skipping
[6/245] CDK2_HUMAN/6Q4E: already exists, skipping
[7/245] CDK2_HUMAN/6Q4K: already exists, skipping
[8/245] CDK2_HUMAN/6Q4D: already exists, skipping
[9/245] CDK2_HUMAN/6Q3B: already exists, skipping
[10/245] CDK2_HUMAN/6Q4I: already exists, skipping
[11/245] CDK2_HUMAN/6Q4B: already exists, skipping
[12/245] CDK2_HUMAN/6Q4A: already exists, skipping
[13/245] CDK2_HUMAN/9GNO: already exists, skipping
[14/245] CDK2_HUMAN/6Q3F: already exists, skipping
[15/245] CDK2_HUMAN/6Q4F: already exists, skipping
[16/245] CDK2_HUMAN/4EK4: already exists, skipping
[17/245] CDK2_HUMAN/4FKL: already exists, skipping
[18/245] CDK2_HUMAN/2R3I: already exists, skipping
[19/245] CDK2_HUMAN/6Q3C: already exists, ski

## 5. 化合物ごとの活性値(pChEMBL)を集計する

セクション2でランキングした蛋白(ChEMBL targetが見つかったもの)について、ChEMBLの活性データを
化合物(構造標準化済み)×標的蛋白の単位で集計し、pChEMBL値のmedian/mean/standard deviation/個数を求める。
median(代表値) >= 9.0(高活性)の組み合わせだけを抽出する。


In [18]:
from molstd.standardize import standardize_smiles

ACTIVITY_RECORDS_CACHE = OUTDIR / "activity_records.pkl"

if ACTIVITY_RECORDS_CACHE.exists():
    print(f"Using cached activity records: {ACTIVITY_RECORDS_CACHE}")
    with open(ACTIVITY_RECORDS_CACHE, "rb") as f:
        activity_records = pickle.load(f)
else:
    activity_records = []
    targets_with_chembl = ranking_df.dropna(subset=["chembl_target_id"])
    total = len(targets_with_chembl)
    for i, row in enumerate(targets_with_chembl.itertuples(), start=1):
        activities = chembl_local.fetch_activities(row.chembl_target_id, CHEMBL_DB)
        print(f"[{i}/{total}] {row.entry_name} ({row.chembl_target_id}): {len(activities)} activities, standardizing ...")
        n_ok = 0
        for a in activities:
            if a["pchembl_value"] is None or a["canonical_smiles"] is None:
                continue
            std_smiles = standardize_smiles(a["canonical_smiles"])
            if std_smiles is None:
                continue
            activity_records.append({
                "smiles": std_smiles,
                "accession": row.accession,
                "entry_name": row.entry_name,
                "pchembl_value": float(a["pchembl_value"]),
            })
            n_ok += 1
        print(f"    -> {n_ok} usable records")

    with open(ACTIVITY_RECORDS_CACHE, "wb") as f:
        pickle.dump(activity_records, f)
    print(f"Saved {len(activity_records)} activity records to {ACTIVITY_RECORDS_CACHE}")

Using cached activity records: data/cdk20_investigation/activity_records.pkl


In [22]:
PCHEMBL_MEDIAN_CUTOFF = 9.0

activity_records_df = pd.DataFrame(activity_records)
print(f"activity records (standardized SMILES, pChEMBL available): {len(activity_records_df)}")

activity_summary_df = (
    activity_records_df
    .groupby(["smiles", "accession", "entry_name"])["pchembl_value"]
    .agg(median="median", mean="mean", std="std", count="count")
    .reset_index()
    .sort_values("median", ascending=False)
    .reset_index(drop=True)
)
activity_summary_df.to_csv(OUTDIR / "activity_summary.csv", index=False)
print(f"compound x target pairs: {len(activity_summary_df)}")

high_potency_df = activity_summary_df[activity_summary_df["median"] >= PCHEMBL_MEDIAN_CUTOFF].reset_index(drop=True)
high_potency_df.to_csv(OUTDIR / "high_potency_compounds.csv", index=False)
print(f"compound x target pairs with median pChEMBL >= {PCHEMBL_MEDIAN_CUTOFF}: {len(high_potency_df)}")

high_potency_display_df = high_potency_df.copy()
for col in ["median", "mean", "std"]:
    high_potency_display_df[col] = high_potency_display_df[col].round(2)
with pd.option_context("display.max_colwidth", None):
    display(high_potency_display_df)

activity records (standardized SMILES, pChEMBL available): 9194
compound x target pairs: 7293
compound x target pairs with median pChEMBL >= 9.0: 45


,smiles,accession,entry_name,median,mean,std,count
0,COc1ccc(F)cc1-c1cc(NC(=O)Cc2cccnc2)ncn1,P50750,CDK9_HUMAN,10.82,10.82,NaN,1
1,COc1ccc(F)cc1-c1cc(NC(=O)C2CCCNC2)ncn1,P50750,CDK9_HUMAN,10.74,10.74,NaN,1
2,COc1ccc(F)cc1-c1cc(NC(=O)Cc2ccncc2)ncn1,P50750,CDK9_HUMAN,10.54,10.54,NaN,1
3,COc1cccc(F)c1-c1cc(NC(=O)C2CCCNC2)ncn1,P50750,CDK9_HUMAN,10.41,10.41,NaN,1
4,COc1ccccc1-c1cc(NC(=O)C2CCC(=O)NC2)ncn1,P50750,CDK9_HUMAN,10.24,10.24,0.68,2
5,NS(=O)(=O)c1ccc(Nc2ncc(C(F)(F)F)c(Nc3ccc4[nH]cnc4c3)n2)cc1,P06493,CDK1_HUMAN,10.00,10.00,NaN,1
6,COc1ccccc1-c1cc(NC(=O)c2ccc(C)c(NS(C)(=O)=O)c2)ncn1,P50750,CDK9_HUMAN,9.94,9.94,NaN,1
7,CC(C)c1cnn2c(NCc3ccccc3-n3cccn3)nc(OC3CCCNC3)nc12,P50613,CDK7_HUMAN,9.89,9.89,NaN,1
8,O=[N+]([O-])c1cccc(Nc2nccc(-c3cnn4ncccc34)n2)c1,P24941,CDK2_HUMAN,9.52,9.52,NaN,1
9,Cn1ncc(-c2nc(N[C@H]3CC[C@H](N)CC3)ncc2F)c1CC1CC1,P50613,CDK7_HUMAN,9.51,9.51,NaN,1


### 5.1 化合物単位に集約する

同じ化合物(標準化SMILES)が複数の標的蛋白に対して活性データを持つ場合があるため、化合物ごとに
グループ化し、テストされた標的数(`target_count`)・最も高い活性(median pChEMBL)を示した標的の
Entry name(`best_target_entry_name`)・その活性値(`best_pchembl_median`)に集約する。


In [ ]:
from molstd import calc_mol_weight

target_counts = activity_summary_df.groupby("smiles").size().rename("target_count").reset_index()

best_rows = activity_summary_df.loc[activity_summary_df.groupby("smiles")["median"].idxmax()][
    ["smiles", "entry_name", "median"]
].rename(columns={"entry_name": "best_target_entry_name", "median": "best_pchembl_median"})

compound_summary_df = best_rows.merge(target_counts, on="smiles").sort_values(
    "best_pchembl_median", ascending=False
).reset_index(drop=True)
compound_summary_df["mol_weight"] = compound_summary_df["smiles"].apply(calc_mol_weight)
compound_summary_df = compound_summary_df[
    ["smiles", "mol_weight", "best_target_entry_name", "best_pchembl_median", "target_count"]
]
compound_summary_df.to_csv(OUTDIR / "compound_summary.csv", index=False)
print(f"unique compounds: {len(compound_summary_df)}")

MOL_WEIGHT_MIN, MOL_WEIGHT_MAX = 250, 650

high_potency_compound_df = compound_summary_df[
    (compound_summary_df["best_pchembl_median"] >= PCHEMBL_MEDIAN_CUTOFF)
    & (compound_summary_df["mol_weight"] >= MOL_WEIGHT_MIN)
    & (compound_summary_df["mol_weight"] <= MOL_WEIGHT_MAX)
].reset_index(drop=True)
high_potency_compound_df.to_csv(OUTDIR / "high_potency_compound_summary.csv", index=False)
print(
    f"unique compounds with best_pchembl_median >= {PCHEMBL_MEDIAN_CUTOFF} "
    f"and {MOL_WEIGHT_MIN} <= mol_weight <= {MOL_WEIGHT_MAX}: {len(high_potency_compound_df)}"
)

high_potency_compound_display_df = high_potency_compound_df.copy()
high_potency_compound_display_df["best_pchembl_median"] = high_potency_compound_display_df["best_pchembl_median"].round(2)
high_potency_compound_display_df["mol_weight"] = high_potency_compound_display_df["mol_weight"].round(2)

# 表示だけ全行・SMILES非トランケートにする。オプションはこのセル内に限定する
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(high_potency_compound_display_df)
